# Technology Presets: Platform-Specific Simulations

Different spatial transcriptomics platforms have distinct characteristics. PointillSim provides **technology presets** that configure simulations to match specific platforms.

## Supported Platforms

| Platform | Type | Gene Panel | Resolution |
|----------|------|------------|------------|
| **HybISS** | FISH | 50-100 | Subcellular |
| **Cartana** | ISS | 100-300 | Subcellular |
| **MERFISH** | FISH | 100-500 | Subcellular |
| **10x Visium** | Sequencing | Whole transcriptome | 55μm spots |
| **10x Xenium** | ISS | 300-1000 | Subcellular |

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.collections import PatchCollection
import pandas as pd

np.random.seed(42)

from pointillsim import (
    TissueCellTypes,
    CellTypesProperties,
    HybISS_Setup,
    FOVDistribution,
    FrameWideElement,
    VacuolatedStructure,
    RandomCellTypeRule,
    MixOfNCellTypesRule,
    DistanceBasedRule,
)

# Import technology presets
from pointillsim.experiment import (
    TechnologyPreset,
    HybISSPreset,
    CartanaPreset,
    MerfishPreset,
    TenXVisiumPreset,
    TenXXeniumPreset,
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'
plt.style.use('seaborn-v0_8-whitegrid')

---
## 1. Understanding Technology Presets

Each preset encapsulates platform-specific parameters.

In [ ]:
# Create all presets
hybiss = HybISSPreset()
cartana = CartanaPreset()
merfish = MerfishPreset()
visium = TenXVisiumPreset()
xenium = TenXXeniumPreset()

presets = {
    'HybISS': hybiss,
    'Cartana': cartana,
    'MERFISH': merfish,
    'Visium': visium,
    'Xenium': xenium,
}

# Display preset parameters
print("Technology Preset Comparison:")
print("=" * 70)

for name, preset in presets.items():
    print(f"\n{name}:")
    print(f"  Max genes: {preset.max_genes}")
    print(f"  Detection efficiency: {preset.detection_efficiency:.1%}")
    print(f"  Spatial resolution: {preset.spatial_resolution} μm")
    print(f"  Typical FOV size: {preset.typical_fov_size} μm")

In [ ]:
# Create comparison table
comparison_data = []
for name, preset in presets.items():
    comparison_data.append({
        'Platform': name,
        'Max Genes': preset.max_genes,
        'Detection Eff.': f"{preset.detection_efficiency:.0%}",
        'Resolution (μm)': preset.spatial_resolution,
        'FOV Size (μm)': preset.typical_fov_size,
        'Type': preset.platform_type,
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nPlatform Comparison:")
comparison_df

---
## 2. Creating Platform-Specific Simulations

Each preset provides a configured `HybISS_Setup` with appropriate parameters.

In [ ]:
# Create a common tissue and FOV
n_cell_types = 5

# Tissue with 100 genes (max allowed by most platforms)
tissue = TissueCellTypes()
tissue.generate_types_and_markers(
    n_genes=100,
    n_cell_types=n_cell_types,
    expected_level=15.0,
    concentration=0.85,
)

cell_props = CellTypesProperties(n_cell_types=n_cell_types)

# Create FOV distribution
frame_size = 500
fov_dist = FOVDistribution(
    frame_size=frame_size,
    background_element=lambda: FrameWideElement(
        frame_size=frame_size,
        tipical_cell_spacing=15,
        rules=MixOfNCellTypesRule(
            n_cell_types=n_cell_types,
            list_N=[0, 1, 2],
            proportions=[0.5, 0.3, 0.2]
        )
    ),
    other_elements=[
        lambda: VacuolatedStructure(
            frame_size=frame_size,
            scale=80,
            hole_scale_factor=0.4,
            tipical_cell_spacing=10,
            rules=DistanceBasedRule(
                n_cell_types=n_cell_types,
                inner_type=3,
                outer_type=4,
            )
        )
    ],
    elements_frequency=[1.0],
    attempts_at_elements=3,
)

# Generate reference FOV
np.random.seed(42)
fov = fov_dist.generate_fov()
cell_props.apply(fov)

print(f"Reference FOV: {fov.n_cells} cells")

In [ ]:
# Generate simulations for each platform
platform_results = {}

for name, preset in presets.items():
    np.random.seed(42)
    
    # Create experiment setup from preset
    hybiss_setup = preset.create_setup(tissue)
    
    # Generate observations
    hybiss_setup.observe_dots(fov)
    dots_df = hybiss_setup.make_pandas_df()
    
    platform_results[name] = {
        'dots_df': dots_df,
        'n_dots': len(dots_df),
        'setup': hybiss_setup,
    }
    
    print(f"{name}: {len(dots_df):,} dots detected")

In [ ]:
# Visualize platform differences
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Reference cells
ax = axes[0]
scatter = ax.scatter(
    fov.cell_centroids[:, 0],
    fov.cell_centroids[:, 1],
    c=fov.class_instance,
    cmap='Set1',
    s=25, alpha=0.8
)
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Ground Truth\n({fov.n_cells} cells)')
ax.set_xticks([])
ax.set_yticks([])

# Each platform
platform_colors = {
    'HybISS': 'darkblue',
    'Cartana': 'darkgreen',
    'MERFISH': 'purple',
    'Visium': 'orange',
    'Xenium': 'darkred',
}

for i, (name, results) in enumerate(platform_results.items()):
    ax = axes[i + 1]
    dots_df = results['dots_df']
    
    ax.scatter(
        dots_df['x'],
        dots_df['y'],
        s=1, alpha=0.4,
        c=platform_colors[name]
    )
    ax.set_xlim(0, frame_size)
    ax.set_ylim(0, frame_size)
    ax.set_aspect('equal')
    ax.set_title(f'{name}\n({len(dots_df):,} dots)')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Same Tissue, Different Technologies', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Platform-Specific Characteristics

Let's explore the unique characteristics of each platform.

In [ ]:
# Compare gene detection across platforms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dots per platform
ax = axes[0]
names = list(platform_results.keys())
dot_counts = [platform_results[n]['n_dots'] for n in names]
colors = [platform_colors[n] for n in names]

bars = ax.bar(names, dot_counts, color=colors, alpha=0.8)
ax.set_ylabel('Total Transcripts')
ax.set_title('Detection Yield by Platform')
ax.tick_params(axis='x', rotation=45)

# Add value labels
for bar, count in zip(bars, dot_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{count:,}', ha='center', va='bottom', fontsize=10)

# Gene distribution
ax = axes[1]
for name, color in platform_colors.items():
    dots_df = platform_results[name]['dots_df']
    gene_counts = dots_df['gene'].value_counts().values
    ax.hist(gene_counts, bins=30, alpha=0.5, label=name, color=color)

ax.set_xlabel('Counts per Gene')
ax.set_ylabel('Number of Genes')
ax.set_title('Gene Count Distribution')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Transcripts per cell by platform
fig, ax = plt.subplots(figsize=(10, 6))

transcripts_per_cell = []
for name in names:
    dots_df = platform_results[name]['dots_df']
    cell_counts = dots_df.groupby('cell').size()
    transcripts_per_cell.append(cell_counts.values)

# Box plot
bp = ax.boxplot(transcripts_per_cell, labels=names, patch_artist=True)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Transcripts per Cell')
ax.set_title('Cell-Level Detection Comparison')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 4. Visium: Spot-Based Technology

Visium is fundamentally different - it uses large capture spots rather than single-molecule detection.

In [ ]:
# Simulate Visium spot layout
visium_preset = TenXVisiumPreset()

# Visium parameters
spot_diameter = visium_preset.spot_diameter  # μm
spot_spacing = visium_preset.spot_spacing    # μm center-to-center

# Generate spot grid
def generate_visium_spots(frame_size, spot_spacing, spot_diameter, pixel_size=1.0):
    """Generate Visium-like spot positions."""
    # Convert to pixel units
    spacing_px = spot_spacing / pixel_size
    radius_px = (spot_diameter / 2) / pixel_size
    
    # Hexagonal grid
    spots = []
    row = 0
    y = radius_px
    
    while y < frame_size - radius_px:
        x_offset = spacing_px / 2 if row % 2 == 1 else 0
        x = radius_px + x_offset
        
        while x < frame_size - radius_px:
            spots.append((x, y, radius_px))
            x += spacing_px
        
        y += spacing_px * np.sqrt(3) / 2
        row += 1
    
    return np.array(spots)

# Generate spots (scaled for visualization)
spots = generate_visium_spots(frame_size, spot_spacing=55, spot_diameter=55, pixel_size=1.0)

print(f"Visium spot layout: {len(spots)} spots")
print(f"  Spot diameter: {spot_diameter} μm")
print(f"  Spot spacing: {spot_spacing} μm")

In [ ]:
# Visualize Visium vs subcellular technologies
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Ground truth cells
ax = axes[0]
scatter = ax.scatter(
    fov.cell_centroids[:, 0],
    fov.cell_centroids[:, 1],
    c=fov.class_instance,
    cmap='Set1',
    s=25, alpha=0.8
)
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title('Ground Truth Cells')
ax.set_xticks([])
ax.set_yticks([])

# Xenium (subcellular)
ax = axes[1]
dots_xenium = platform_results['Xenium']['dots_df']
ax.scatter(dots_xenium['x'], dots_xenium['y'], s=1, alpha=0.4, c='darkred')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title('Xenium: Single-Molecule Resolution')
ax.set_xticks([])
ax.set_yticks([])

# Visium (spot-based)
ax = axes[2]

# Draw spots
circles = []
spot_colors = []
for x, y, r in spots:
    # Count cells in this spot
    dist = np.sqrt((fov.cell_centroids[:, 0] - x)**2 + (fov.cell_centroids[:, 1] - y)**2)
    cells_in_spot = np.sum(dist < r)
    circles.append(Circle((x, y), r))
    spot_colors.append(cells_in_spot)

collection = PatchCollection(circles, alpha=0.7, cmap='YlOrRd')
collection.set_array(np.array(spot_colors))
ax.add_collection(collection)

# Overlay cell positions
ax.scatter(fov.cell_centroids[:, 0], fov.cell_centroids[:, 1],
           c='black', s=5, alpha=0.3)

ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Visium: 55μm Spots\n({len(spots)} spots, avg {np.mean(spot_colors):.1f} cells/spot)')
ax.set_xticks([])
ax.set_yticks([])
plt.colorbar(collection, ax=ax, label='Cells per spot')

plt.suptitle('Resolution Comparison: Single-Molecule vs Spot-Based', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Customizing Presets

You can modify preset parameters for specific experiments.

In [ ]:
# Create a custom MERFISH-like preset with modified parameters
class CustomMerfishPreset(MerfishPreset):
    """MERFISH with increased noise for challenging benchmarks."""
    
    def __init__(self):
        super().__init__()
        # Increase noise
        self.detection_efficiency = 0.5  # Lower than default
        self.background_rate = 0.002     # Higher background
        self.error_rate = 0.05           # Higher error rate

# Create comparison
standard_merfish = MerfishPreset()
noisy_merfish = CustomMerfishPreset()

print("Standard MERFISH vs Noisy MERFISH:")
print(f"  Detection efficiency: {standard_merfish.detection_efficiency:.0%} vs {noisy_merfish.detection_efficiency:.0%}")
print(f"  Background rate: {standard_merfish.background_rate:.3f} vs {noisy_merfish.background_rate:.3f}")
print(f"  Error rate: {standard_merfish.error_rate:.2%} vs {noisy_merfish.error_rate:.2%}")

In [ ]:
# Generate with both presets
np.random.seed(42)
standard_setup = standard_merfish.create_setup(tissue)
standard_setup.observe_dots(fov)
dots_standard = standard_setup.make_pandas_df()

np.random.seed(42)
noisy_setup = noisy_merfish.create_setup(tissue)
noisy_setup.observe_dots(fov)
dots_noisy = noisy_setup.make_pandas_df()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(dots_standard['x'], dots_standard['y'], s=1, alpha=0.4, c='purple')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Standard MERFISH\n({len(dots_standard):,} dots)')
ax.set_xticks([])
ax.set_yticks([])

ax = axes[1]
ax.scatter(dots_noisy['x'], dots_noisy['y'], s=1, alpha=0.4, c='darkmagenta')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Noisy MERFISH\n({len(dots_noisy):,} dots)')
ax.set_xticks([])
ax.set_yticks([])

plt.suptitle('Custom Preset: Increasing Technical Noise', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nDot reduction: {100*(1 - len(dots_noisy)/len(dots_standard)):.1f}%")

---
## 6. Complete Platform Comparison

Generate a comprehensive comparison across all platforms.

In [ ]:
# Comprehensive comparison
fig, axes = plt.subplots(3, 2, figsize=(12, 15))

# 1. Transcript count comparison
ax = axes[0, 0]
platform_names = list(platform_results.keys())
counts = [platform_results[p]['n_dots'] for p in platform_names]
bars = ax.barh(platform_names, counts, color=[platform_colors[p] for p in platform_names], alpha=0.8)
ax.set_xlabel('Total Transcripts')
ax.set_title('Detection Yield')
for bar, count in zip(bars, counts):
    ax.text(count + 500, bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', fontsize=9)

# 2. Transcripts per cell
ax = axes[0, 1]
mean_per_cell = []
for p in platform_names:
    dots_df = platform_results[p]['dots_df']
    cell_counts = dots_df.groupby('cell').size()
    mean_per_cell.append(cell_counts.mean())

bars = ax.barh(platform_names, mean_per_cell, color=[platform_colors[p] for p in platform_names], alpha=0.8)
ax.set_xlabel('Mean Transcripts per Cell')
ax.set_title('Per-Cell Detection')
for bar, val in zip(bars, mean_per_cell):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

# 3. Gene detection breadth
ax = axes[1, 0]
genes_detected = []
for p in platform_names:
    dots_df = platform_results[p]['dots_df']
    genes_detected.append(dots_df['gene'].nunique())

bars = ax.barh(platform_names, genes_detected, color=[platform_colors[p] for p in platform_names], alpha=0.8)
ax.set_xlabel('Genes Detected')
ax.set_title('Gene Coverage')
for bar, val in zip(bars, genes_detected):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=9)

# 4. Gene count variability
ax = axes[1, 1]
for p in platform_names:
    dots_df = platform_results[p]['dots_df']
    gene_counts = dots_df['gene'].value_counts()
    ax.hist(gene_counts, bins=20, alpha=0.5, label=p, color=platform_colors[p])

ax.set_xlabel('Counts per Gene')
ax.set_ylabel('Number of Genes')
ax.set_title('Gene Count Distribution')
ax.legend(fontsize=8)

# 5. Spatial dot density
ax = axes[2, 0]
for i, p in enumerate(platform_names):
    dots_df = platform_results[p]['dots_df']
    density = len(dots_df) / (frame_size * frame_size) * 1e4  # per 100x100 px
    ax.bar(i, density, color=platform_colors[p], alpha=0.8)

ax.set_xticks(range(len(platform_names)))
ax.set_xticklabels(platform_names, rotation=45, ha='right')
ax.set_ylabel('Dots per 100x100 pixels')
ax.set_title('Spatial Dot Density')

# 6. Platform characteristics summary
ax = axes[2, 1]
ax.axis('off')

summary_text = """
Platform Characteristics Summary
================================

FISH-based (HybISS, MERFISH):
  • High detection efficiency
  • Limited gene panel (50-500)
  • Subcellular resolution

ISS-based (Cartana, Xenium):
  • Rolling circle amplification
  • Larger gene panels possible
  • Good for tissue sections

Sequencing-based (Visium):
  • Whole transcriptome
  • Lower spatial resolution
  • Spot-based (55μm)
  • Multiple cells per spot
"""

ax.text(0.1, 0.9, summary_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Comprehensive Platform Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

| Preset | Best For | Limitations |
|--------|----------|-------------|
| **HybISSPreset** | Small targeted panels, high sensitivity | Limited genes |
| **CartanaPreset** | In situ sequencing studies | Moderate throughput |
| **MerfishPreset** | Large panels, high multiplexing | Complex workflow |
| **TenXVisiumPreset** | Discovery, whole transcriptome | Low resolution (spot) |
| **TenXXeniumPreset** | Large panels, commercial platform | Fixed panels |

### Key Takeaways

1. **Presets simplify** platform-specific simulation configuration
2. **Detection yield varies** significantly across platforms
3. **Visium is fundamentally different** (spot-based vs single-molecule)
4. **Custom presets** allow fine-tuning for specific benchmarks

### Next Steps

- **12_beautiful_gallery.ipynb**: Gallery of complete tissue simulations